In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import cv2
import os
import glob
import math
import json
import shutil
from sklearn.model_selection import train_test_split
import pydicom
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.layers import GlobalAveragePooling2D
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
# Global variables and paths

DATA_DIR = "data"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
MODEL_SAVE_PATH = "models/pneumonia_vgg16.h5"

os.makedirs("models", exist_ok=True)

In [ ]:
def load_preprocess_dicom_vgg16(file_path, augment=False):
    """
    Loads a DICOM format X-ray image and prepares it for the VGG16 network.

    Args:
        file_path (str): The absolute or relative path to the .dcm file.
        augment (bool, optional): Whether to apply random data augmentation (blur, noise). Defaults to False.

    Returns:
        np.ndarray: The 224x224, RGB channel, VGG16-optimized image array (float32).
    """

    dicom = pydicom.dcmread(file_path)
    img = dicom.pixel_array.astype(np.float32)

    # Normalize pixel values to 0-255
    img = img - np.min(img)

    if np.max(img) != 0:
        img = img / np.max(img)

    img = img * 255.0
    img = img.astype(np.uint8)

    # Resize input image to 224x224 for VGG16
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    if augment and np.random.rand() < 0.5: # 50% chance to apply augmentation
        choice = np.random.rand()
        if choice < 0.5:
            # Applying blur to the image
            img = cv2.GaussianBlur(img, (5, 5), 0)
        else:
            # Applying noise to the image
            noise = np.random.normal(0, 10, img.shape).astype(np.float32)
            img = np.clip(img + noise, 0, 255).astype(np.uint8)
    
    # Grayscale to RGB
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    img = img.astype(np.float32)

    # VGG16 preprocessing
    img = preprocess_input(img)

    return img

In [ ]:
def collect_paths_and_labels(split):
    """
    Collects file paths and corresponding labels for a specified dataset split.

    Args:
        split (str): The name of the folder split ('train', 'val', 'test').

    Returns:
        tuple: (paths, labels) as numpy arrays. Labels: 0 (not_pneumonia), 1 (pneumonia).
    """

    pneumonia_paths = glob.glob(os.path.join(DATA_DIR, split, "pneumonia", "*.dcm"))
    normal_paths = glob.glob(os.path.join(DATA_DIR, split, "not_pneumonia", "*.dcm"))

    paths = pneumonia_paths + normal_paths
    labels = [1] * len(pneumonia_paths) + [0] * len(normal_paths)

    paths = np.array(paths)
    labels = np.array(labels)

    return paths, labels


train_paths, train_labels = collect_paths_and_labels("train")
val_paths, val_labels = collect_paths_and_labels("val")
test_paths, test_labels = collect_paths_and_labels("test")

print("Train images:", len(train_paths))
print("Validation images:", len(val_paths))
print("Test images:", len(test_paths))

print("Train class count:", np.bincount(train_labels))
print("Val class count:", np.bincount(val_labels))
print("Test class count:", np.bincount(test_labels))

In [ ]:

class DicomDataGenerator(Sequence):
    """
    Custom Keras sequence generator for memory-efficient and batched loading of DICOM files.
    """

    def __init__(self, paths, labels, batch_size=32, shuffle=True, augment=False, **kwargs):
        """
        Initializes the data generator.

        Args:
            paths (np.ndarray): Array of file paths to the images.
            labels (np.ndarray): Array of corresponding class labels.
            batch_size (int, optional): Number of samples per batch. Defaults to 32.
            shuffle (bool, optional): Whether to shuffle the data at the end of each epoch. Defaults to True.
            augment (bool, optional): Whether to apply image augmentations. Defaults to False.
        """
        super().__init__(**kwargs)
        self.paths = paths
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.indices = np.arange(len(self.paths))
        self.on_epoch_end()

    def __len__(self):
        """Returns the number of batches per epoch."""
        return math.ceil(len(self.paths) / self.batch_size)

    def __getitem__(self, idx):
        """Generates and returns one batch of data (images and labels)."""
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]

        batch_paths = self.paths[batch_indices]
        batch_labels = self.labels[batch_indices]

        images = []

        for path in batch_paths:
            img = load_preprocess_dicom_vgg16(path, augment=self.augment)
            images.append(img)

        images = np.array(images, dtype=np.float32)
        labels = np.array(batch_labels, dtype=np.float32)

        return images, labels

    def on_epoch_end(self):
        """Shuffles indexes after each epoch if shuffle is set to True."""
        if self.shuffle:
            np.random.shuffle(self.indices)

In [ ]:
# Create data generators for training, validation, and testing datasets

train_gen = DicomDataGenerator(
    train_paths,
    train_labels,
    batch_size=BATCH_SIZE,
    shuffle=True,
    augment=True) # Only the training generator has augmentation enabled

val_gen = DicomDataGenerator(
    val_paths,
    val_labels,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_gen = DicomDataGenerator(
    test_paths,
    test_labels,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
# Test one batch from the training generator to verify shapes and labels

x_batch, y_batch = train_gen[0]

print("Batch image shape:", x_batch.shape)
print("Batch label shape:", y_batch.shape)
print("Batch labels:", y_batch[:10])

In [ ]:
# Create the VGG16-based model

base_model = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

base_model.trainable = True

for layer in base_model.layers[:-4]:
    layer.trainable = False

model = models.Sequential([
    base_model,
    GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Compute class weights to handle class imbalance

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = {
    0: class_weights_array[0],
    1: class_weights_array[1]
}

print("Class weights:", class_weights)

In [ ]:
# Training the model with callbacks for early stopping, learning rate reduction, model checkpointing

checkpoint = ModelCheckpoint(
    filepath='models/forsafety.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

early_stopper = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.4,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stopper, checkpoint, reduce_lr]
)

In [ ]:
model.save(MODEL_SAVE_PATH)

print("Model saved to:", MODEL_SAVE_PATH)

In [ ]:
def plot_training_history(history):
    """
    Plots the training metrics (Loss and Accuracy) across epochs.

    Args:
        history (tf.keras.callbacks.History): The history object returned by model.fit().
    """
    plt.figure(figsize=(12, 5))
    
    # Accuracy Plot
    plt.subplot(1, 2, 1)
    plt.plot(history.history["accuracy"], label="Training accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training and validation accuracy")
    plt.legend()
    
    # Loss Plot
    plt.subplot(1, 2, 2)
    plt.plot(history.history["loss"], label="Training loss")
    plt.plot(history.history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and validation loss")
    plt.legend()
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Plot training and validation metrics
plot_training_history(history)

In [ ]:
# Evaluate the model on the test dataset

test_loss, test_accuracy = model.evaluate(test_gen)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

In [ ]:
def predict_and_compare_dicom(file_path, model, true_label, threshold=0.5):
    """
    Predicts pneumonia, displays the X-ray, and compares the prediction with the ground truth.

    Args:
        file_path (str): The path to the .dcm file.
        model (tf.keras.Model): The trained classification model.
        true_label (str or int): The actual ground truth ("pneumonia"/"not_pneumonia" or 1/0).
        threshold (float, optional): The probability threshold. Defaults to 0.5.
    """
    # Preprocess and predict
    img = load_preprocess_dicom_vgg16(file_path)
    img_batch = np.expand_dims(img, axis=0)
    prediction = model.predict(img_batch, verbose=0)[0][0]

    # Convert prediction to string label
    pred_label = "pneumonia" if prediction > threshold else "not_pneumonia"

    # Standardize true_label to string for comparison
    if isinstance(true_label, (int, np.integer)):
        actual_label = "pneumonia" if true_label == 1 else "not_pneumonia"
    else:
        actual_label = str(true_label).lower().strip()

    # Check if the model was correct
    is_correct = (pred_label == actual_label)
    title_color = "green" if is_correct else "red"

    # Print to console
    print(f"Correct: {is_correct} | True: {actual_label} | Pred: {pred_label} (Prob: {prediction:.4f})")

    # X-ray display
    plt.figure(figsize=(6, 6))

    img_original = img[:, :, 0] 
    
    plt.imshow(img_original, cmap="gray") 
        
    plt.title(
        f"TRUE: {actual_label.upper()} \nPRED: {pred_label.upper()} ({prediction:.4f})\n"
        f"Result: {'CORRECT' if is_correct else 'WRONG'}", 
        fontsize=12, 
        color=title_color,
        fontweight="bold"
    )
    plt.axis("off")
    plt.show()

    return pred_label, prediction

In [ ]:

example_path = test_paths[0]
example_true_label = test_labels[0]
    
predict_and_compare_dicom(example_path, model, true_label=example_true_label, threshold=0.5)

In [ ]:

def evaluate_model_with_thresholds(model, test_gen, test_labels, thresholds=[0.5]):
    """
    Evaluates the model by printing a detailed classification report and 
    plotting a visual confusion matrix for one or more thresholds.

    Args:
        model (tf.keras.Model): The trained Keras model.
        test_gen (DicomDataGenerator): The generator containing the test dataset.
        test_labels (np.ndarray): The ground truth labels for the test set.
        thresholds (list, optional): List of probability thresholds to evaluate.
    """
    # Generate predictions once
    predictions = model.predict(test_gen)
    num_thresholds = len(thresholds)
    
    fig, axes = plt.subplots(1, num_thresholds, figsize=(6 * num_thresholds, 5))
    
    # Ensure axes is iterable even for a single threshold
    if num_thresholds == 1:
        axes = [axes]
    
    for i, threshold in enumerate(thresholds):
        # Convert probabilities to binary class predictions
        y_pred = (predictions > threshold).astype(int).reshape(-1)
        
        # Print classification report
        print(f"Threshold: {threshold}")
        print(classification_report(
            test_labels, 
            y_pred, 
            target_names=["Not Pneumonia", "Pneumonia"]
        ))
        
        # Compute confusion matrix
        cm = confusion_matrix(test_labels, y_pred)
        
        # Display the confusion matrix
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not P.", "Pneumonia"])
        disp.plot(ax=axes[i], cmap=plt.cm.Blues, values_format='d', colorbar=False)
        axes[i].set_title(f"Confusion Matrix (Threshold: {threshold})")
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot confusion matrices for multiple thresholds
thresholds = [0.5, 0.4, 0.3, 0.2]

evaluate_model_with_thresholds(model, test_gen, test_labels, thresholds=thresholds)